# Pipeline ICMP → X

Este documento apresenta o o caminho completo que os dados percorrem no projeto:

**pacote ICMP (echo request/reply) → 1 registro → janela (n registros) → X = [latência, perda, jitter]**

**Dispositivos envolvidos:**
- **Dispositivo que executa o teste:** este notebook, rodando no Google Colab.
- **Dispositivo de destino:** `8.8.8.8` (servidor DNS público do Google, sempre responde).

Essa é apenas **uma** das fontes de dados possíveis para o projeto — as outras são datasets reais já coletados e a API pública do RIPE Atlas.

In [7]:
!pip install icmplib pandas -q

from icmplib import ping
from datetime import datetime, timezone
import pandas as pd
import time

# Destinos monitorados: IP -> quantas "janelas" (rodadas de teste) fazer nele
destinos = {
    "8.8.8.8": 30,        # Google DNS - rápido e estável -> tende a OK
    "1.1.1.1": 30,        # Cloudflare DNS - rápido e estável -> tende a OK
    "180.76.76.76": 30,   # Baidu DNS (China) - longe -> tende a RISCO
    "192.0.2.1": 20,      # Endereço reservado, nunca responde -> sempre FALHA
}

def coletar_janela(destino, n=8, intervalo=0.15, timeout=1.0):
    registros = []
    for _ in range(n):
        resultado = ping(destino, count=1, timeout=timeout, privileged=False)
        respondeu = 1 if resultado.packets_received == 1 else 0
        rtt_ms = resultado.rtts[0] if respondeu else None
        registros.append({"respondeu": respondeu, "rtt_ms": rtt_ms})
        time.sleep(intervalo)
    return pd.DataFrame(registros)

def calcular_metricas(df):
    respondidos = df[df["respondeu"] == 1]
    total = len(df)
    perda_pct = 100 * (1 - len(respondidos) / total)
    latencia_ms = respondidos["rtt_ms"].mean() if len(respondidos) > 0 else None
    if len(respondidos) >= 2:
        jitter_ms = respondidos["rtt_ms"].diff().abs().dropna().mean()
    else:
        jitter_ms = 0.0
    return latencia_ms, perda_pct, jitter_ms

def classificar(latencia_ms, perda_pct):
    if latencia_ms is None or perda_pct >= 15:
        return "FALHA"
    if latencia_ms > 150 or perda_pct >= 2:
        return "RISCO"
    return "OK"

linhas = []

for ip, num_janelas in destinos.items():
    for j in range(num_janelas):
        df_janela = coletar_janela(ip)
        latencia_ms, perda_pct, jitter_ms = calcular_metricas(df_janela)
        status = classificar(latencia_ms, perda_pct)
        linhas.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "ip": ip,
            "latencia_ms": round(latencia_ms, 2) if latencia_ms is not None else None,
            "perda_pacotes_pct": round(perda_pct, 2),
            "jitter_ms": round(jitter_ms, 2),
            "status_real": status
        })
        print(f"{ip} | janela {j+1}/{num_janelas} | {status}")

df_log = pd.DataFrame(linhas)
df_log.to_csv("log_rede.csv", index=False)
print("\nSalvo em log_rede.csv -", len(df_log), "linhas")
print(df_log["status_real"].value_counts())

8.8.8.8 | janela 1/30 | OK
8.8.8.8 | janela 2/30 | OK
8.8.8.8 | janela 3/30 | OK
8.8.8.8 | janela 4/30 | OK
8.8.8.8 | janela 5/30 | OK
8.8.8.8 | janela 6/30 | OK
8.8.8.8 | janela 7/30 | OK
8.8.8.8 | janela 8/30 | OK
8.8.8.8 | janela 9/30 | OK
8.8.8.8 | janela 10/30 | OK
8.8.8.8 | janela 11/30 | OK
8.8.8.8 | janela 12/30 | OK
8.8.8.8 | janela 13/30 | OK
8.8.8.8 | janela 14/30 | OK
8.8.8.8 | janela 15/30 | OK
8.8.8.8 | janela 16/30 | OK
8.8.8.8 | janela 17/30 | OK
8.8.8.8 | janela 18/30 | OK
8.8.8.8 | janela 19/30 | OK
8.8.8.8 | janela 20/30 | OK
8.8.8.8 | janela 21/30 | OK
8.8.8.8 | janela 22/30 | OK
8.8.8.8 | janela 23/30 | OK
8.8.8.8 | janela 24/30 | OK
8.8.8.8 | janela 25/30 | OK
8.8.8.8 | janela 26/30 | OK
8.8.8.8 | janela 27/30 | OK
8.8.8.8 | janela 28/30 | OK
8.8.8.8 | janela 29/30 | OK
8.8.8.8 | janela 30/30 | OK
1.1.1.1 | janela 1/30 | OK
1.1.1.1 | janela 2/30 | OK
1.1.1.1 | janela 3/30 | OK
1.1.1.1 | janela 4/30 | RISCO
1.1.1.1 | janela 5/30 | OK
1.1.1.1 | janela 6/30 | OK
1.1.

In [8]:
import pandas as pd
import random
from datetime import datetime, timedelta, timezone

random.seed(42)  # garante que o resultado seja sempre igual, se rodar de novo

def gerar_registro(status):
    if status == "OK":
        latencia_ms = round(random.uniform(5, 100), 2)
        perda_pacotes_pct = round(random.uniform(0, 1.5), 2)
        jitter_ms = round(random.uniform(0, 10), 2)
    elif status == "RISCO":
        latencia_ms = round(random.uniform(100, 300), 2)
        perda_pacotes_pct = round(random.uniform(2, 14), 2)
        jitter_ms = round(random.uniform(10, 40), 2)
    else:  # FALHA
        latencia_ms = round(random.uniform(300, 800), 2)
        perda_pacotes_pct = round(random.uniform(15, 100), 2)
        jitter_ms = round(random.uniform(40, 150), 2)
    return latencia_ms, perda_pacotes_pct, jitter_ms

ips_fake = ["10.0.0.1", "10.0.0.2", "10.0.0.3", "10.0.0.4"]
TOTAL_POR_CLASSE = 50  # 50 OK + 50 RISCO + 50 FALHA = 150 linhas

linhas = []
inicio = datetime.now(timezone.utc)

for status in ["OK", "RISCO", "FALHA"]:
    for i in range(TOTAL_POR_CLASSE):
        latencia_ms, perda_pacotes_pct, jitter_ms = gerar_registro(status)
        linhas.append({
            "timestamp": (inicio + timedelta(seconds=i*5)).isoformat(),
            "ip": random.choice(ips_fake),
            "latencia_ms": latencia_ms,
            "perda_pacotes_pct": perda_pacotes_pct,
            "jitter_ms": jitter_ms,
            "status_real": status
        })

random.shuffle(linhas)
df_sintetico = pd.DataFrame(linhas)
df_sintetico.to_csv("dataset_sintetico.csv", index=False)
print("Salvo em dataset_sintetico.csv -", len(df_sintetico), "linhas")
print(df_sintetico["status_real"].value_counts())

Salvo em dataset_sintetico.csv - 150 linhas
status_real
RISCO    50
OK       50
FALHA    50
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score
import pandas as pd

# Carrega o dataset sintético (usa o que já está na memória, ou lê do arquivo)
df = pd.read_csv("dataset_sintetico.csv")

# Separa as colunas que a árvore vai usar para decidir (X)
# e a coluna que ela precisa aprender a prever (y)
X = df[["latencia_ms", "perda_pacotes_pct", "jitter_ms"]]
y = df["status_real"]

# Divide em treino (70%) e teste (30%)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Cria e treina a árvore de decisão
arvore = DecisionTreeClassifier(max_depth=4, random_state=42)
arvore.fit(X_treino, y_treino)

# Testa a árvore nos dados que ela nunca viu
y_previsto = arvore.predict(X_teste)
acuracia = accuracy_score(y_teste, y_previsto)

print(f"Acurácia no conjunto de teste: {acuracia:.2%}")
print(f"Treino: {len(X_treino)} linhas | Teste: {len(X_teste)} linhas")
print()
print("Como a árvore decide (texto):")
print(export_text(arvore, feature_names=["latencia_ms", "perda_pacotes_pct", "jitter_ms"]))

Acurácia no conjunto de teste: 100.00%
Treino: 105 linhas | Teste: 45 linhas

Como a árvore decide (texto):
|--- perda_pacotes_pct <= 2.10
|   |--- class: OK
|--- perda_pacotes_pct >  2.10
|   |--- jitter_ms <= 40.60
|   |   |--- class: RISCO
|   |--- jitter_ms >  40.60
|   |   |--- class: FALHA



In [10]:
from collections import deque
import pandas as pd

# Carrega o log real
df_log = pd.read_csv("log_rede.csv")

N_HISTORICO = 5  # quantas últimas medições guardar por IP

# A TABELA HASH: um dicionário onde a chave é o IP
# e o valor é uma "fila" com as últimas N medições daquele IP
tabela_hash = {}

def registrar_medicao(ip, status, latencia_ms, perda_pacotes_pct, jitter_ms):
    if ip not in tabela_hash:
        tabela_hash[ip] = deque(maxlen=N_HISTORICO)  # cria o histórico desse IP, se não existir
    tabela_hash[ip].append({
        "status": status,
        "latencia_ms": latencia_ms,
        "perda_pacotes_pct": perda_pacotes_pct,
        "jitter_ms": jitter_ms
    })

# Alimenta a tabela hash com cada linha do log, em ordem
for _, linha in df_log.iterrows():
    registrar_medicao(
        linha["ip"],
        linha["status_real"],
        linha["latencia_ms"],
        linha["perda_pacotes_pct"],
        linha["jitter_ms"]
    )

# Mostra o histórico de cada IP (só as últimas N medições de cada um)
for ip, historico in tabela_hash.items():
    print(f"\nIP: {ip}  —  últimas {len(historico)} medições:")
    for medicao in historico:
        print(f"  status: {medicao['status']:6}  latência: {medicao['latencia_ms']}ms  perda: {medicao['perda_pacotes_pct']}%")


IP: 8.8.8.8  —  últimas 5 medições:
  status: OK      latência: 0.5ms  perda: 0.0%
  status: OK      latência: 0.53ms  perda: 0.0%
  status: OK      latência: 0.59ms  perda: 0.0%
  status: OK      latência: 0.5ms  perda: 0.0%
  status: OK      latência: 0.55ms  perda: 0.0%

IP: 1.1.1.1  —  últimas 5 medições:
  status: RISCO   latência: 14.03ms  perda: 12.5%
  status: OK      latência: 21.14ms  perda: 0.0%
  status: OK      latência: 16.91ms  perda: 0.0%
  status: RISCO   latência: 19.48ms  perda: 12.5%
  status: OK      latência: 27.48ms  perda: 0.0%

IP: 180.76.76.76  —  últimas 5 medições:
  status: RISCO   latência: 190.02ms  perda: 0.0%
  status: RISCO   latência: 189.42ms  perda: 0.0%
  status: RISCO   latência: 188.85ms  perda: 0.0%
  status: RISCO   latência: 190.38ms  perda: 0.0%
  status: RISCO   latência: 189.94ms  perda: 0.0%

IP: 192.0.2.1  —  últimas 5 medições:
  status: FALHA   latência: nanms  perda: 100.0%
  status: FALHA   latência: nanms  perda: 100.0%
  status: FA

## 1. Instalar e importar

In [1]:
%pip install icmplib pandas -q

from icmplib import ping
from datetime import datetime
import pandas as pd
import time

DESTINO = "8.8.8.8"


## 2. Um pacote ICMP = um registro

Enviamos **um** echo request. Se o destino responder, guardamos o RTT (em ms). Se não responder dentro do timeout, guardamos `None` — nunca zero, porque zero seria um tempo de resposta real, e aqui o que temos é ausência de resposta.

In [2]:
def uma_tentativa(destino, sequencia, timeout=2):
    resultado = ping(destino, count=1, interval=0.2, timeout=timeout, privileged=False)
    respondeu = 1 if resultado.packets_received == 1 else 0
    rtt_ms = float(resultado.rtts[0]) if respondeu else None

    return {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "destino": destino,
        "sequencia": sequencia,
        "respondeu": respondeu,
        "rtt_ms": rtt_ms
    }

# Um único registro
uma_tentativa(DESTINO, 1)


{'timestamp': '2026-10-03T22:03:08',
 'destino': '8.8.8.8',
 'sequencia': 1,
 'respondeu': 1,
 'rtt_ms': 0.8082389831542969}

## 3. Uma janela = várias tentativas seguidas

Uma tentativa isolada não diz quase nada sobre a qualidade da rede. Precisamos de **várias tentativas em sequência** — isso é uma janela de medição.

In [3]:
def coletar_janela(destino, n=10, intervalo=0.2):
    registros = []
    for seq in range(1, n + 1):
        registros.append(uma_tentativa(destino, seq))
        time.sleep(intervalo)
    return pd.DataFrame(registros)

janela = coletar_janela(DESTINO, n=10)
janela


,timestamp,destino,sequencia,respondeu,rtt_ms
0,2026-10-03T22:03:08,8.8.8.8,1,1,0.420570
1,2026-10-03T22:03:08,8.8.8.8,2,1,0.577211
2,2026-10-03T22:03:09,8.8.8.8,3,1,0.476360
3,2026-10-03T22:03:09,8.8.8.8,4,1,0.528097
4,2026-10-03T22:03:09,8.8.8.8,5,1,0.513554
5,2026-10-03T22:03:09,8.8.8.8,6,1,0.563622
6,2026-10-03T22:03:09,8.8.8.8,7,1,0.489473
7,2026-10-03T22:03:10,8.8.8.8,8,1,0.492096
8,2026-10-03T22:03:10,8.8.8.8,9,1,1.754284
9,2026-10-03T22:03:10,8.8.8.8,10,1,0.401974


## 4. Da janela para X = [latência, perda, jitter]

- **Latência:** média dos RTTs válidos (ignorando os `None`).
- **Perda:** percentual de tentativas sem resposta.
- **Jitter:** média das diferenças absolutas entre RTTs consecutivos válidos — mede o quanto o RTT varia de um pacote para o outro.

In [4]:
def calcular_X(df):
    rtts = df["rtt_ms"].dropna()

    latencia = float(rtts.mean()) if len(rtts) > 0 else None
    perda = (len(df) - df["respondeu"].sum()) / len(df) * 100
    jitter = float(rtts.diff().abs().mean()) if len(rtts) >= 2 else None

    return [latencia, perda, jitter]

X = calcular_X(janela)
print("X = [latência_ms, perda_%, jitter_ms] =", X)


X = [latência_ms, perda_%, jitter_ms] = [0.6217241287231445, np.float64(0.0), 0.3405676947699653]


## 5. Repetindo para várias janelas
Um vetor X sozinho ainda não forma um dataset. Repetindo a coleta várias vezes, cada janela vira **uma linha** — é esse formato que alimenta o modelo preditor mais adiante no projeto.

In [5]:
dataset = []

NUM_JANELAS = 3
TAMANHO_JANELA = 10

for j in range(1, NUM_JANELAS + 1):
    df_janela = coletar_janela(DESTINO, n=TAMANHO_JANELA)
    latencia, perda, jitter = calcular_X(df_janela)
    dataset.append({
        "janela": j,
        "latencia_ms": latencia,
        "perda_pct": perda,
        "jitter_ms": jitter
    })

df_dataset = pd.DataFrame(dataset)
df_dataset


,janela,latencia_ms,perda_pct,jitter_ms
0,1,0.744367,0.0,0.419299
1,2,1.149035,0.0,0.874440
2,3,0.648284,0.0,0.199848


## 6. Salvar o resultado

In [6]:
df_dataset.to_csv("dataset_features.csv", index=False)
print("Salvo em dataset_features.csv")


Salvo em dataset_features.csv


## Recapitulando o pipeline

```
Echo Request/Reply (ICMP)
        │
        ▼
1 registro  → uma_tentativa()      → timestamp, destino, sequência, respondeu, rtt_ms
        │
        ▼
janela (n registros) → coletar_janela()
        │
        ▼
X = [latência, perda, jitter]  → calcular_X()
        │
        ▼
dataset (uma linha por janela)
```

Este notebook usa `icmplib` rodando no próprio Colab como **uma** fonte de dados. No projeto completo, essa mesma estrutura de X também pode ser alimentada por **datasets reais de medição de rede** já publicados, ou pela **API do RIPE Atlas**, que usa uma rede global de probes em vez de um único dispositivo. O formato de saída (X) é o mesmo — muda apenas de onde os dados brutos vêm.